# Programa B — Validación del dataset

## Propósito

Antes de entrenar un modelo debemos comprobar que el conjunto de datos sea coherente.

Validaremos:

- existencia de archivos;
- cantidad de imágenes;
- etiquetas faltantes;
- etiquetas inválidas;
- registros duplicados;
- imágenes sin etiqueta;
- etiquetas sin imagen;
- distribución de las clases.


## 1. Importar librerías


In [1]:
# Si fuera necesario instalar dependencias:
# %pip install pandas pillow matplotlib

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image


## 2. Definir las rutas


In [2]:
BASE = Path(".")
CARPETA_IMAGENES = BASE / "imagenes"
ARCHIVO_ETIQUETAS = CARPETA_IMAGENES / "etiquetas.csv"

ETIQUETAS_VALIDAS = {"circulo", "cuadrado", "triangulo"}

print("Carpeta de imágenes:", CARPETA_IMAGENES.resolve())
print("Archivo de etiquetas:", ARCHIVO_ETIQUETAS.resolve())


Carpeta de imágenes: /home/alerrsi/Documents/ML/eva2/evaluacion-2-machine-learning/imagenes
Archivo de etiquetas: /home/alerrsi/Documents/ML/eva2/evaluacion-2-machine-learning/imagenes/etiquetas.csv


## 3. Cargar el archivo de etiquetas


In [3]:
df = pd.read_csv(ARCHIVO_ETIQUETAS)
display(df.head())

print("\nCantidad de registros:", len(df))
print("Columnas:", list(df.columns))


,archivo,etiqueta
0,img_001.png,circulo
1,img_002.png,circulo
2,img_003.png,circulo
3,img_004.png,circulo
4,img_005.png,cuadrado



Cantidad de registros: 91
Columnas: ['archivo', 'etiqueta']


## 4. Verificar columnas obligatorias

El archivo debe contener al menos:

- `archivo`
- `etiqueta`


In [4]:
columnas_obligatorias = {"archivo", "etiqueta"}
faltan_columnas = columnas_obligatorias - set(df.columns)

if faltan_columnas:
    print("Faltan columnas obligatorias:", faltan_columnas)
else:
    print("Las columnas obligatorias están presentes.")


Las columnas obligatorias están presentes.


## 5. Buscar valores faltantes


In [5]:
nulos = df[["archivo", "etiqueta"]].isna().sum()
display(nulos)

if nulos.sum() == 0:
    print("No se encontraron valores faltantes.")
else:
    print("Existen valores faltantes que deben revisarse.")


archivo     0
etiqueta    3
dtype: int64

Existen valores faltantes que deben revisarse.


## 6. Buscar registros duplicados

Una misma imagen no debería aparecer más de una vez en el archivo de etiquetas.


In [6]:
duplicados = df[df.duplicated(subset="archivo", keep=False)]

if duplicados.empty:
    print("No existen imágenes duplicadas en etiquetas.csv.")
else:
    print("Se encontraron registros duplicados:")
    display(duplicados.sort_values("archivo"))


No existen imágenes duplicadas en etiquetas.csv.


## 7. Verificar etiquetas válidas


In [7]:
etiquetas_encontradas = set(df["etiqueta"].dropna().astype(str))
invalidas = etiquetas_encontradas - ETIQUETAS_VALIDAS

print("Etiquetas encontradas:", sorted(etiquetas_encontradas))

if invalidas:
    print("Etiquetas inválidas:", sorted(invalidas))
else:
    print("Todas las etiquetas son válidas.")


Etiquetas encontradas: ['circulo', 'cuadrado', 'desconocido', 'triangulo']
Etiquetas inválidas: ['desconocido']


## 8. Comparar imágenes físicas con el CSV

No basta con que exista el registro en el CSV. También debe existir realmente el archivo de imagen.


In [ ]:
extensiones = {".png", ".jpg", ".jpeg", ".bmp", ".gif"}

imagenes_en_carpeta = {
    p.name
    for p in CARPETA_IMAGENES.iterdir()
    if p.is_file() and p.suffix.lower() in extensiones
}

imagenes_en_csv = set(df["archivo"].dropna().astype(str))

sin_etiqueta = imagenes_en_carpeta - imagenes_en_csv
sin_imagen = imagenes_en_csv - imagenes_en_carpeta

print("Imágenes físicas encontradas:", len(imagenes_en_carpeta))
print("Imágenes registradas en CSV:", len(imagenes_en_csv))

print("\nImágenes sin etiqueta:")
print(sorted(sin_etiqueta) if sin_etiqueta else "Ninguna")

print("\nRegistros CSV sin archivo de imagen:")
print(sorted(sin_imagen) if sin_imagen else "Ninguno")


## 9. Verificar que las imágenes puedan abrirse

Un archivo puede existir y aun así estar dañado o no ser una imagen válida.


In [ ]:
imagenes_con_error = []

for nombre in sorted(imagenes_en_carpeta):
    ruta = CARPETA_IMAGENES / nombre
    try:
        with Image.open(ruta) as img:
            img.verify()
    except Exception as error:
        imagenes_con_error.append((nombre, str(error)))

if not imagenes_con_error:
    print("Todas las imágenes pueden abrirse correctamente.")
else:
    print("Se encontraron imágenes con problemas:")
    for nombre, error in imagenes_con_error:
        print(nombre, "->", error)


## 10. Analizar la distribución de clases

Un conjunto equilibrado contiene cantidades similares de ejemplos por clase.

Esto no siempre es obligatorio, pero conviene conocer la distribución antes de entrenar.


In [ ]:
distribucion = df["etiqueta"].value_counts().sort_index()
display(distribucion)

distribucion.plot(kind="bar")
plt.title("Cantidad de imágenes por clase")
plt.xlabel("Clase")
plt.ylabel("Cantidad")
plt.xticks(rotation=0)
plt.show()


## 11. Resumen automático de validación


In [ ]:
problemas = []

if faltan_columnas:
    problemas.append("Faltan columnas obligatorias.")

if nulos.sum() > 0:
    problemas.append("Existen valores faltantes.")

if not duplicados.empty:
    problemas.append("Existen registros duplicados.")

if invalidas:
    problemas.append("Existen etiquetas inválidas.")

if sin_etiqueta:
    problemas.append("Existen imágenes sin etiqueta.")

if sin_imagen:
    problemas.append("Existen etiquetas cuyo archivo de imagen no existe.")

if imagenes_con_error:
    problemas.append("Existen imágenes que no pueden abrirse.")

if problemas:
    print("EL DATASET REQUIERE REVISIÓN:")
    for problema in problemas:
        print("-", problema)
else:
    print("EL DATASET SUPERÓ LAS VALIDACIONES BÁSICAS.")


## 12. Preguntas de análisis

1. ¿Por qué una imagen sin etiqueta no puede utilizarse directamente en aprendizaje supervisado?
2. ¿Qué problema produce una etiqueta cuyo archivo de imagen no existe?
3. ¿Qué podría ocurrir si una clase tiene muchos más ejemplos que las otras?
4. ¿Por qué conviene detectar duplicados antes del entrenamiento?
5. ¿Validar un dataset garantiza que sus etiquetas sean correctas? Justifique.


## Idea clave

**Validar** un dataset no significa demostrar que es perfecto.

Significa comprobar sistemáticamente que su estructura, archivos y valores cumplen las condiciones necesarias para poder trabajar con ellos.
